In [44]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import torch
from PIL import Image
from transformers import ViTImageProcessor, ViTForImageClassification
from transformers import AutoImageProcessor, AutoModel
#from transformers import CLIPProcessor, CLIPModel
import pandas as pd
import numpy as np
import os
import pickle
import glob
import requests


## Embedding

In [6]:
path = "D:\\Research"

In [7]:
artnet_2024 = pd.read_excel(f"{path}\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_merged.xlsx")

In [8]:
artnet_2024.shape

(355551, 16)

In [13]:
#number_size = int(sys.argv[1])
#range_start = int(sys.argv[2])
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)

In [23]:
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

In [24]:
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
model = AutoModel.from_pretrained('facebook/dinov2-base')

In [28]:
inputs = processor(images=image, return_tensors="pt")
outputs = model(**inputs)
embedding = outputs.last_hidden_state[:, 0, :]

In [35]:
embedding/ embedding.norm(p=2, dim=-1, keepdim=True)

tensor([[-4.6809e-02, -9.5615e-03,  2.3323e-02, -2.8272e-02, -6.3118e-04,
          1.6079e-02, -3.2855e-04,  3.3544e-02, -4.1715e-02,  5.1867e-02,
         -9.3179e-03, -2.7750e-02,  2.3491e-02,  3.0148e-02, -1.9693e-02,
         -3.5290e-02,  5.0708e-02,  1.1458e-03, -1.8474e-03,  4.0994e-02,
          8.5082e-03, -2.6767e-02, -6.1761e-02,  4.9751e-02,  1.0698e-02,
         -2.8439e-02, -1.6347e-02,  1.0478e-02, -3.2092e-02,  1.0868e-02,
         -2.8357e-03, -4.5188e-02, -3.9709e-02, -5.2765e-02,  3.5953e-02,
          1.3546e-02, -1.2725e-02, -3.4123e-02,  7.4634e-03,  4.6015e-03,
         -3.4811e-02,  8.9718e-02, -1.7428e-02, -4.2952e-03,  3.5380e-02,
          4.6257e-02,  4.5088e-03,  2.0987e-02, -3.9510e-02, -2.4540e-02,
         -4.4481e-02, -2.9992e-02,  8.9715e-03, -8.6658e-04, -1.3921e-02,
          2.5168e-02,  9.0154e-03,  9.7212e-03,  6.7008e-03, -1.6910e-02,
          2.7122e-02, -8.6729e-03,  2.6673e-02,  1.8924e-03, -2.0965e-02,
         -2.1063e-02, -1.1098e-02,  2.

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [36]:
def convert_one_row(model,i, image_file,device):
    try:
        image = Image.open(f"{path}\\Creativity_Artnet\\Datasets\\ArtNet\\Images\\{image_file}")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model(**inputs).last_hidden_state[:, 0, :]
        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        image_features = image_features.cpu().numpy()
    except Exception as e:
        print(f"Error processing {image_file}: {e}")
        image_features = np.zeros([1,512])
        
    return i, image_features

In [ ]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
embeddings = np.zeros(N, dtype=object)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(convert_one_row, model,i, f'{artnet_2024.iloc[i]["artwork id"]}.jpg',device): i
        for i in range(range_start,range_end)
    }
    for fut in as_completed(futures):
        i, image_features = fut.result()
        embeddings[i-range_start] =image_features
        if i % 10000 == 0:
            print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

In [ ]:
np.save(f"Result/clip_embeddings_{range_start}.npy", embeddings)

In [42]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < artnet_2024.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    embeddings = np.zeros(N, dtype=object)
    with ThreadPoolExecutor(max_workers=32) as ex:
        futures = {
            ex.submit(convert_one_row, model,i, f'{artnet_2024.iloc[i]["artwork id"]}.jpg',device): i
            for i in range(range_start,range_end)
        }
        for fut in as_completed(futures):
            i, image_features = fut.result()
            embeddings[i-range_start] =image_features
            if i % 10000 == 0:
                print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
    np.save(f"{path}\\Creativity_Artnet\\Datasets\\Dinov2_Embedding_2024\\dinov2_embeddings_{range_start}.npy", embeddings)
    range_start = range_start + N
    range_end = min(range_start+number_size,artnet_2024.shape[0])
    N = min(number_size, artnet_2024.shape[0]-range_start)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2026-01-01 22:19:20: Start
2026-01-01 22:19:20: Now at 0
2026-01-01 22:19:26: 0
2026-01-01 22:25:05: 10000


The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


Error processing 424340448.jpg: mean must have 1 elements if it is an iterable, got 3
2026-01-01 22:31:18: 20000
2026-01-01 22:37:38: 30000
2026-01-01 22:43:57: 40000
2026-01-01 22:50:19: 50000
2026-01-01 22:56:37: 60000
2026-01-01 23:02:30: 70000
2026-01-01 23:08:59: 80000
2026-01-01 23:15:26: 90000
2026-01-01 23:21:56: Saving embeddings
2026-01-01 23:21:56: Now at 100000
2026-01-01 23:22:19: 100000
2026-01-01 23:28:35: 110000
2026-01-01 23:34:54: 120000
2026-01-01 23:40:42: 130000
2026-01-01 23:46:50: 140000
2026-01-01 23:52:54: 150000
2026-01-01 23:58:59: 160000
2026-01-02 00:05:03: 170000
2026-01-02 00:11:08: 180000
2026-01-02 00:16:57: 190000
2026-01-02 00:23:02: Saving embeddings
2026-01-02 00:23:02: Now at 200000
2026-01-02 00:23:26: 200000
2026-01-02 00:29:26: 210000
2026-01-02 00:33:43: 220000
2026-01-02 00:36:22: 230000
2026-01-02 00:38:59: 240000
2026-01-02 00:41:37: 250000
2026-01-02 00:44:18: 260000
2026-01-02 00:46:57: 270000
2026-01-02 00:49:30: 280000


The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


Error processing 440635219.jpg: mean must have 1 elements if it is an iterable, got 3
2026-01-02 00:52:06: 290000
2026-01-02 00:54:40: Saving embeddings
2026-01-02 00:54:40: Now at 300000
2026-01-02 00:54:50: 300000
2026-01-02 00:57:20: 310000
2026-01-02 00:59:57: 320000
2026-01-02 01:02:30: 330000
2026-01-02 01:05:06: 340000
2026-01-02 01:07:45: 350000


The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). A

Error processing 444379706.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379735.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379736.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379754.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379818.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379767.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379799.jpg: mean must have 1 elements if it is an iterable, got 3


The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). A

Error processing 444383507.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383511.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383525.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383531.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383532.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383529.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383542.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383543.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383536.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383561.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383568.jpg: mean must have 1 elements if it is an iterable, got 3
2026-01-02 01:09:10: Saving embeddings
2026-01-02 01:0

In [10]:
artnet_2025.shape

(371778, 26)

## Merge all Outputs

In [45]:
folder_path =  f"{path}\\Creativity_Artnet\\Datasets\\Dinov2_Embedding_2024"
npy_files = glob.glob(f"{folder_path}/*.npy")

In [46]:
arrays = [np.load(f, allow_pickle=True) for f in npy_files]
big_array = np.concatenate(arrays, axis=0)
np.save(f"{path}\\Creativity_Artnet\\Datasets\\Dinov2_Embedding_2024.npy", big_array)

In [47]:
def fix_embedding(e):
    # Ensure numpy array
    e = np.asarray(e)

    for row in range(e.shape[0]):
    # Case: (1, 512) and all zeros
        if e[row].shape == (1, 512) and np.all(e[row] == 0):
            e[row]=np.zeros((1, 768), dtype=e.dtype)

    # Otherwise leave untouched
    return e

In [49]:
a = fix_embedding(big_array)

In [50]:
a= np.vstack(a)

In [53]:
a[0]

array([-0.024541286751627922, -0.05906897410750389, 0.019260583445429802,
       0.004685881081968546, 0.058799032121896744, -0.010986192151904106,
       0.007753307931125164, 0.03903190791606903, 0.0148876141756773,
       -0.06808032840490341, 0.02820313535630703, -0.06585084646940231,
       -0.034895818680524826, -0.033622074872255325, -0.07016017287969589,
       0.03729954734444618, 0.018656501546502113, 0.011612915433943272,
       -0.015722401440143585, -0.012961757369339466, 0.009396946057677269,
       0.0052479892037808895, 0.01987224631011486, -0.08329325914382935,
       -0.04017620161175728, 0.008991878479719162, -0.028551990166306496,
       0.0353519432246685, -0.03381992504000664, -0.01983872801065445,
       -0.012949768453836441, 0.02130209654569626, -0.06250619888305664,
       0.0013325867475941777, -0.028127675876021385, -0.05785635486245155,
       -0.008674184791743755, -0.0262703076004982, 0.007377625908702612,
       -0.04666728898882866, -0.00876067671924829

In [52]:
np.save(f"{path}\\Creativity_Artnet\\Datasets\\Dinov2_Embedding_2024.npy", a)

# Formal 2024

In [4]:
artnet_2024 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_merged.xlsx")

In [5]:
artnet_2024.shape

(355551, 16)

In [ ]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2025.shape[0])
N = min(number_size, artnet_2025.shape[0]-range_start)

In [6]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32",use_fast=True)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [8]:
def convert_one_row(model,i, image_file,device):
    try:
        image = Image.open(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Images\\{image_file}")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        image_features = image_features.cpu().numpy()
    except Exception as e:
        print(f"Error processing {image_file}: {e}")
        image_features = np.zeros([1,512])
        
    return i, image_features

In [ ]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
embeddings = np.zeros(N, dtype=object)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(convert_one_row, model,i, f'{artnet_2025.iloc[i]["artwork id"]}.jpg',device): i
        for i in range(range_start,range_end)
    }
    for fut in as_completed(futures):
        i, image_features = fut.result()
        embeddings[i-range_start] =image_features
        if i % 10000 == 0:
            print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

In [ ]:
np.save(f"Result/clip_embeddings_{range_start}.npy", embeddings)

In [10]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < artnet_2024.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    embeddings = np.zeros(N, dtype=object)
    with ThreadPoolExecutor(max_workers=32) as ex:
        futures = {
            ex.submit(convert_one_row, model,i, f'{artnet_2024.iloc[i]["artwork id"]}.jpg',device): i
            for i in range(range_start,range_end)
        }
        for fut in as_completed(futures):
            i, image_features = fut.result()
            embeddings[i-range_start] =image_features
            if i % 10000 == 0:
                print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
    np.save(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Embedding_2024\\clip_embeddings_{range_start}.npy", embeddings)
    range_start = range_start + N
    range_end = min(range_start+number_size,artnet_2024.shape[0])
    N = min(number_size, artnet_2024.shape[0]-range_start)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-12-08 03:21:23: Start
2025-12-08 03:21:23: Now at 0
2025-12-08 03:21:34: 0
2025-12-08 03:22:55: 10000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


2025-12-08 03:24:17: 20000
2025-12-08 03:25:38: 30000
2025-12-08 03:26:59: 40000
2025-12-08 03:28:19: 50000
2025-12-08 03:29:41: 60000
2025-12-08 03:31:02: 70000
2025-12-08 03:32:22: 80000
2025-12-08 03:33:43: 90000
2025-12-08 03:35:05: Saving embeddings
2025-12-08 03:35:05: Now at 100000
2025-12-08 03:35:52: 100000
2025-12-08 03:36:39: 110000
2025-12-08 03:38:01: 120000
2025-12-08 03:39:24: 130000
2025-12-08 03:40:47: 140000
2025-12-08 03:42:10: 150000
2025-12-08 03:43:33: 160000
2025-12-08 03:44:55: 170000
2025-12-08 03:46:18: 180000
2025-12-08 03:47:40: 190000
2025-12-08 03:49:03: Saving embeddings
2025-12-08 03:49:03: Now at 200000
2025-12-08 03:49:40: 200000
2025-12-08 03:50:38: 210000
2025-12-08 03:52:01: 220000
2025-12-08 03:53:23: 230000
2025-12-08 03:54:46: 240000
2025-12-08 03:56:08: 250000
2025-12-08 03:57:31: 260000
2025-12-08 03:58:53: 270000
2025-12-08 04:00:16: 280000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


2025-12-08 04:01:38: 290000
2025-12-08 04:03:02: Saving embeddings
2025-12-08 04:03:02: Now at 300000
2025-12-08 04:03:41: 300000
2025-12-08 04:04:31: 310000
2025-12-08 04:05:53: 320000
2025-12-08 04:07:16: 330000
2025-12-08 04:08:39: 340000
2025-12-08 04:10:01: 350000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is amb

2025-12-08 04:10:47: Saving embeddings
2025-12-08 04:10:47: Ends


# Merging

In [11]:
folder_path = "D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Embedding_2024"
npy_files = glob.glob(f"{folder_path}/*.npy")

In [12]:
arrays = [np.load(f, allow_pickle=True) for f in npy_files]
big_array = np.concatenate(arrays, axis=0)
np.save("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\clip_embedding_2024.npy", big_array)